# Simulated Out-of-Sample Test

In [ ]:
import sys, os
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from stable_baselines3 import PPO

from evaluate import rollout, ZScorePolicy

import warnings; warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({'figure.dpi':110, 'axes.grid':True,
                     'grid.alpha':0.3, 'axes.spines.top':False,
                     'axes.spines.right':False})

PAIR_NAME = 'V_MA'
TRANSACTION_COST = 0.001
ZSCORE_WINDOW = 20
TEST_SEED = 999       # must differ from training SEED=42
N_TEST = 500
CALIB_ALPHA = 0.05

REGIME_NAMES  = ['Stable', 'Neutral', 'Crisis']
REGIME_COLORS = ['#2196F3', '#FF9800', '#E91E63']

BETA = float(np.load(f'./regime_output_{PAIR_NAME}/beta.npy')[0])
print(f'BETA={BETA:.4f}')

## 1. Load Calibration Artifacts

In [ ]:
ou_path = f'./regime_output_{PAIR_NAME}/ou_params.pkl'
if not os.path.exists(ou_path):
    raise FileNotFoundError(
        f'{ou_path} not found. Re-run the generator notebook '
    )

with open(ou_path, 'rb') as f:
    p = pickle.load(f)

kappa        = p['kappa']
mu_ou        = p['mu_ou']
sigma_global = p['sigma_global']
sigma_r      = p['sigma_r']
trans        = p['trans']
init_spread  = p['init_spread']
DT           = p['DT']
N_REGIMES    = p['N_REGIMES']
HORIZON      = p['HORIZON']

mu_r    = p.get('mu_r',    {r: mu_ou for r in range(N_REGIMES)})
kappa_r = p.get('kappa_r', {r: kappa for r in range(N_REGIMES)})


## 2. Generate Held-Out Test Episodes

In [ ]:
np.random.seed(TEST_SEED)

PERTURB_SIGMA = 0.20

cum_trans = np.cumsum(trans, axis=1)

e_g_r = {r: float(np.exp(-kappa_r[r] * DT)) for r in range(N_REGIMES)}

spreads_test = np.zeros((N_TEST, HORIZON), dtype=np.float32)
regimes_test = np.zeros((N_TEST, HORIZON), dtype=np.int8)


import joblib
from numpy.linalg import eig as _eig
best_hmm = joblib.load(f'./regime_output_{PAIR_NAME}/hmm_model.pkl')
_ev, _evec = _eig(best_hmm.transmat_.T)
_idx = np.argmin(np.abs(_ev - 1.0))
_st  = np.real(_evec[:, _idx])
stationary_dist = (_st / _st.sum())

spreads_test[:, 0] = init_spread
regimes_test[:, 0] = np.random.choice(
    N_REGIMES, size=N_TEST, p=stationary_dist/stationary_dist.sum()
)

for ep in range(N_TEST):
   
    sigma_ep = {r: sigma_r[r] * np.random.uniform(1-PERTURB_SIGMA, 1+PERTURB_SIGMA)
                for r in range(N_REGIMES)}
    c_std_ep = {r: float(np.sqrt((sigma_ep[r]**2 / (2*kappa_r[r]))
                                  * (1 - e_g_r[r]**2)))
                for r in range(N_REGIMES)}

    for t in range(1, HORIZON):
        prev_r = int(regimes_test[ep, t-1])
        rnd    = np.random.rand()
        new_r  = int((rnd > cum_trans[prev_r, :-1]).sum())
        regimes_test[ep, t] = new_r
        c_mean = (spreads_test[ep, t-1] * e_g_r[new_r]
                  + mu_r[new_r] * (1 - e_g_r[new_r]))
        spreads_test[ep, t] = float(c_mean + np.random.normal(0, c_std_ep[new_r]))


print(f'Generated {N_TEST} test episodes x {HORIZON} steps')
print(f'Per-episode sigma range: +-{PERTURB_SIGMA*100:.0f}% of calibrated values')

## 3. Load Trained Models

In [ ]:
import glob, re

def _resolve(d):
    """Pick best_model if present, else final, inside a model dir; drop .zip."""
    best  = f'{d}/best_model.zip'
    final = f'{d}/final.zip'
    return (best if os.path.exists(best) else final).replace('.zip', '')

def load_seed_models(label):
    """All per-seed models for ensembling. Returns [(seed, model), ...].
    Falls back to the old flat layout (seed 0) if no seed* subdirs exist."""
    seed_dirs = sorted(glob.glob(f'models_{PAIR_NAME}_sim/{label}/seed*'),
                       key=lambda d: int(re.search(r'seed(\d+)', d).group(1)))
    if not seed_dirs:
        d = f'models_{PAIR_NAME}_sim/{label}'
        m = PPO.load(_resolve(d)); m.verbose = 0
        print(f'  {label:<12}: flat layout -> 1 model')
        return [(0, m)]
    out = []
    for d in seed_dirs:
        s = int(re.search(r'seed(\d+)', d).group(1))
        m = PPO.load(_resolve(d)); m.verbose = 0
        out.append((s, m))
    print(f'  {label:<12}: {len(out)} seed models -> seeds {[s for s, _ in out]}')
    return out

print(f'Obs dim (regime, first seed): '
      f'{load_seed_models("regime")[0][1].observation_space.shape[0]}')

## 4. Multi-Seed Testing

In [ ]:
import hashlib

ens_base = load_seed_models('baseline')
ens_reg  = load_seed_models('regime')

REG_EXTRA = dict(mu_r=mu_r, sigma_r=sigma_r, kappa=kappa)

def seed_curves_and_metrics(ens, use_regimes, **extra):
    curves, rows = [], []
    for s, m in ens:
        r   = rollout(m, spreads_test, regimes_test,
                      use_regimes=use_regimes, beta=BETA, **extra)
        cap = r['capital'] / r['capital'][:, :1] * 100
        curves.append(cap.mean(axis=0))           # MEAN across episodes
        rows.append({
            'Return %': r['returns'].mean(),
            'Sharpe':   r['sharpes'].mean(),
            'Max DD %': r['max_dds'].mean(),
            'Trades':   r['n_trades'].mean(),
        })
    return np.array(curves), pd.DataFrame(rows)


CACHE_PATH      = f'./regime_output_{PAIR_NAME}/multiseed_results.pkl'
FORCE_RECOMPUTE = False   # set True to ignore the cache and rerun the rollouts

def _arr_hash(*arrays):
    h = hashlib.md5()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()

def _file_hash(*paths):
    h = hashlib.md5()
    for p in paths:
        with open(p, 'rb') as f:
            h.update(f.read())
    return h.hexdigest()

cache_key = {
    'pair':        PAIR_NAME,
    'spreads':     _arr_hash(spreads_test),
    'regimes':     _arr_hash(regimes_test),
    'beta':        round(float(BETA), 8),
    'calib_alpha': CALIB_ALPHA,
    'tc':          TRANSACTION_COST,
    'zwin':        ZSCORE_WINDOW,
    'base_seeds':  [s for s, _ in ens_base],
    'reg_seeds':   [s for s, _ in ens_reg],
    'code':        _file_hash('evaluate.py', 'env.py'),
}

def _compute_multiseed():
    cb, mb = seed_curves_and_metrics(ens_base, use_regimes=False)
    cr, mr = seed_curves_and_metrics(ens_reg,  use_regimes=True,
                                     calib_alpha=CALIB_ALPHA, **REG_EXTRA)
    rz = rollout(ZScorePolicy(threshold=1.5), spreads_test, regimes_test,
                 use_regimes=False, beta=BETA)
    cz = (rz['capital'] / rz['capital'][:, :1] * 100).mean(axis=0)
    return dict(curves_base=cb, m_base=mb, curves_reg=cr, m_reg=mr,
                r_z=rz, curve_z=cz)

_data = None
if not FORCE_RECOMPUTE and os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        _blob = pickle.load(f)
    if _blob.get('key') == cache_key:
        _data = _blob['data']
        print(f'[cache] loaded multi-seed results from {CACHE_PATH}')
    else:
        print('[cache] inputs or code changed -> recomputing...')

if _data is None:
    if FORCE_RECOMPUTE:
        print('[cache] FORCE_RECOMPUTE=True -> recomputing...')
    _data = _compute_multiseed()
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump({'key': cache_key, 'data': _data}, f)
    print(f'[cache] saved multi-seed results to {CACHE_PATH}')

curves_base = _data['curves_base']
m_base      = _data['m_base']
curves_reg  = _data['curves_reg']
m_reg       = _data['m_reg']
r_z         = _data['r_z']
curve_z     = _data['curve_z']

steps = np.arange(curves_base.shape[1])


def plot_ensemble(band, title):
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.suptitle(
        f'Simulated Test - Mean Capital Across Training Seeds  ({title})\n'
        f'PPO Baseline: {len(ens_base)} seeds  |  PPO Regime: {len(ens_reg)} seeds  '
        '|  Z-Score: deterministic',
        fontsize=12
    )

    def draw(curves, color, label, ls='-'):
        mean = curves.mean(axis=0)
        if band == 'std':
            sd = curves.std(axis=0)
            lo, hi = mean - sd, mean + sd
        else:  # 'minmax'
            lo, hi = curves.min(axis=0), curves.max(axis=0)
        ax.fill_between(steps, lo, hi, color=color, alpha=0.18)
        ax.plot(steps, mean, color=color, lw=2.0, ls=ls, label=label)

    draw(curves_base, '#2196F3', f'PPO Baseline  (n={len(ens_base)} seeds)')
    draw(curves_reg,  '#E91E63', f'PPO Regime  (n={len(ens_reg)} seeds)')
    ax.plot(steps, curve_z, color='#FF9800', lw=2.0, ls='--', label='Z-Score')

    ax.axhline(100, color='gray', lw=0.8, ls=':', label='Starting capital')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Capital (indexed to 100)')
    ax.legend(fontsize=9, loc='upper left')
    plt.tight_layout()
    plt.show()


plot_ensemble('std',    'band = mean +- 1 SD over seeds')
plot_ensemble('minmax', 'band = seed min–max')

### Multi-Seed Mean Metrics Table

In [ ]:
COLS = ['Return %', 'Sharpe', 'Max DD %', 'Trades']

z_row = pd.Series({
    'Return %': r_z['returns'].mean(),
    'Sharpe':   r_z['sharpes'].mean(),
    'Max DD %': r_z['max_dds'].mean(),
    'Trades':   r_z['n_trades'].mean(),
})

summary = pd.DataFrame({
    f'PPO Baseline (mean of {len(ens_base)} seeds)': m_base.mean(),
    f'PPO Regime (mean of {len(ens_reg)} seeds)':    m_reg.mean(),
    'Z-Score (deterministic)':                       z_row,
}).T[COLS]

summary_sd = pd.DataFrame({
    f'PPO Baseline (mean of {len(ens_base)} seeds)': m_base.std(ddof=1),
    f'PPO Regime (mean of {len(ens_reg)} seeds)':    m_reg.std(ddof=1),
    'Z-Score (deterministic)':                       pd.Series(np.nan, index=COLS),
}).T[COLS]

def _fmt(name, m, sd, dec):
    return f'{m:.{dec}f} ± {sd:.{dec}f}' if np.isfinite(sd) else f'{m:.{dec}f}'

DEC = {'Return %': 2, 'Sharpe': 3, 'Max DD %': 2, 'Trades': 1}
summary_pm = pd.DataFrame({
    c: [_fmt(r, summary.loc[r, c], summary_sd.loc[r, c], DEC[c]) for r in summary.index]
    for c in COLS
}, index=summary.index)


print('Mean test metrics across training seeds value +- 1 SD over seeds'
      '(Z-Score deterministic):')
display(summary_pm)
